In [22]:

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report, confusion_matrix
)

from xgboost import XGBClassifier
import joblib


In [24]:

DATA_PATH = "/content/drive/MyDrive/processed_milestone2_dataset.xlsx"

df = pd.read_excel(DATA_PATH)
print("Dataset Loaded Successfully!")

print(df.shape)
df.head()


Dataset Loaded Successfully!
(10019, 36)


,order_id,supplier_id,supplier_rating,supplier_lead_time,order_date,promised_delivery_date,actual_delivery_date,shipping_distance_km,order_quantity,unit_price,...,region_South,region_West,holiday_period_Yes,carrier_name_DHL,carrier_name_Delhivery,carrier_name_EcomExpress,carrier_name_FedEx,delayed_reason_code_Operational,delayed_reason_code_Traffic,delayed_reason_code_Weather
0,1.0,5322.0,3.4,10,2024-05-15,2024-05-25,2024-05-29,51,48,2153.91,...,False,False,False,False,False,True,False,True,False,False
1,2.0,3932.0,4.3,10,2024-11-12,2024-11-22,2024-11-27,373,91,405.36,...,False,False,True,True,False,False,False,False,False,False
2,3.0,8966.0,3.2,5,2024-08-28,2024-09-02,2024-09-02,1304,25,3241.41,...,True,False,False,False,False,False,False,False,False,False
3,4.0,9832.0,3.9,7,2024-08-12,2024-08-19,2024-08-19,839,71,365.79,...,False,False,False,False,False,False,True,True,False,False
4,5.0,2126.0,3.2,8,2024-07-07,2024-07-15,2024-07-18,258,9,3052.84,...,False,False,False,False,False,False,False,False,False,False


In [25]:
df.drop(columns=["order_id", "supplier_id"], errors="ignore", inplace=True)


In [26]:
df.drop(
    columns=["order_date", "actual_delivery_date", "promised_delivery_date"],
    errors="ignore",
    inplace=True
)


In [27]:
# Force binary target
df["on_time_delivery"] = df["on_time_delivery"].apply(
    lambda x: 1 if x == 1 else 0
).astype(int)

# Verify
df["on_time_delivery"].value_counts()


,count
on_time_delivery,
0,7239
1,2780


In [28]:
X = df.drop("on_time_delivery", axis=1)
y = df["on_time_delivery"]


In [29]:
X = df.drop("on_time_delivery", axis=1)
y = df["on_time_delivery"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [30]:
scale_cols = [
    "delivery_days",
    "order_quantity",
    "shipping_distance_km",
    "total_order_value"
]

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[scale_cols] = scaler.fit_transform(X_train_scaled[scale_cols])
X_test_scaled[scale_cols] = scaler.transform(X_test_scaled[scale_cols])


In [31]:
lr = LogisticRegression(
    max_iter=10000,
    class_weight="balanced",
    solver="saga",
    n_jobs=-1
)

lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]


In [32]:
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]


In [33]:
xgb = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_train.values, y_train.values)

y_pred_xgb = xgb.predict(X_test.values)
y_prob_xgb = xgb.predict_proba(X_test.values)[:, 1]


In [34]:
def evaluate_model(name, y_true, y_pred, y_prob):
    print(f"\n📊 {name}")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred))
    print("ROC-AUC:", roc_auc_score(y_true, y_prob))


In [35]:
evaluate_model("Logistic Regression", y_test, y_pred_lr, y_prob_lr)
evaluate_model("Random Forest", y_test, y_pred_rf, y_prob_rf)
evaluate_model("XGBoost", y_test, y_pred_xgb, y_prob_xgb)



📊 Logistic Regression
[[1040  408]
 [ 122  434]]
              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1448
           1       0.52      0.78      0.62       556

    accuracy                           0.74      2004
   macro avg       0.71      0.75      0.71      2004
weighted avg       0.79      0.74      0.75      2004

ROC-AUC: 0.8232876406057474

📊 Random Forest
[[1427   21]
 [  34  522]]
              precision    recall  f1-score   support

           0       0.98      0.99      0.98      1448
           1       0.96      0.94      0.95       556

    accuracy                           0.97      2004
   macro avg       0.97      0.96      0.97      2004
weighted avg       0.97      0.97      0.97      2004

ROC-AUC: 0.9846799356095235

📊 XGBoost
[[1428   20]
 [  18  538]]
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1448
           1       0.96      0.97      0.97      

In [38]:
print("Sample A:", xgb.predict_proba(X_test.iloc[[1]].values))
print("Sample B:", xgb.predict_proba(X_test.iloc[[10]].values))


Sample A: [[0.98711175 0.01288828]]
Sample B: [[9.9909359e-01 9.0641115e-04]]


In [40]:
import joblib

joblib.dump(xgb, "best_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(X.columns.tolist(), "model_features.pkl")

print("✅ Model, scaler, and feature list saved successfully")


✅ Model, scaler, and feature list saved successfully
